# Verdant-Minds Colab Startup Kit

Idempotent bootstrap for clone/update, dependency install, Drive persistence, resumable runs, and post-run analysis.

## 1) Clone or update repo

In [ ]:
from pathlib import Path
import sys
import subprocess

REPO_URL = 'https://github.com/Captainkoopa42/Verdant-Minds.git'
REPO_DIR = Path('/content/Verdant-Minds')

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print(f'Updating existing repo at {REPO_DIR}...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a git repository.')
else:
    print(f'Cloning into {REPO_DIR}...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', REPO_DIR)

## 2) Install dependencies

In [ ]:
from scripts.colab_bootstrap import install_deps
install_deps(repo_dir=REPO_DIR)

## 3) Mount Drive + configure paths

In [ ]:
from scripts.colab_bootstrap import mount_drive_and_prepare

paths = mount_drive_and_prepare('/content/drive/MyDrive/Verdant')
DRIVE_ROOT = paths['drive_root']
DRIVE_STATE_PATH = paths['state_path']
DRIVE_BASELINE_PATH = paths['baseline_path']
OUTPUTS_ROOT = paths['outputs_root']

print('DRIVE_STATE_PATH =', DRIVE_STATE_PATH)
print('OUTPUTS_ROOT =', OUTPUTS_ROOT)

## 4) Configure provider + run settings (no secrets in plain text)

In [ ]:
import os
import getpass

# Prefer Colab Secrets or runtime env vars. Prompt only if needed.
for key in ['MISTRAL_API_KEY', 'GROQ_API_KEY', 'ANTHROPIC_API_KEY', 'OPENAI_API_KEY']:
    if not os.environ.get(key):
        value = getpass.getpass(f'{key} (leave blank to skip): ')
        if value:
            os.environ[key] = value

# Provider chain for verdant_llm_cultivator.py
os.environ['VERDANT_PROVIDER_CHAIN'] = os.environ.get('VERDANT_PROVIDER_CHAIN', 'mistral,groq,anthropic,openai,local')

# Run configuration
N_RUNS = int(os.environ.get('VERDANT_N_RUNS', '2'))
CYCLES = int(os.environ.get('VERDANT_CYCLES', '40'))
SEED_TOPIC = os.environ.get('VERDANT_SEED_TOPIC', 'contradiction')
PERTURB_INTERVAL = int(os.environ.get('VERDANT_PERTURB_INTERVAL', '10'))

print('VERDANT_PROVIDER_CHAIN =', os.environ['VERDANT_PROVIDER_CHAIN'])
print({'N_RUNS': N_RUNS, 'CYCLES': CYCLES, 'SEED_TOPIC': SEED_TOPIC, 'PERTURB_INTERVAL': PERTURB_INTERVAL})

## 5) Run cultivator N times (resume/fresh)

In [ ]:
from scripts.colab_bootstrap import run_cultivator_loop

env_overrides = {
    'VERDANT_PROVIDER_CHAIN': os.environ.get('VERDANT_PROVIDER_CHAIN', 'mistral,groq,anthropic,openai,local'),
}

run_output_dirs = run_cultivator_loop(
    repo_dir='/content/Verdant-Minds',
    state_path=DRIVE_STATE_PATH,
    n_runs=N_RUNS,
    cycles=CYCLES,
    seed_topic=SEED_TOPIC,
    perturb_interval=PERTURB_INTERVAL,
    env_overrides=env_overrides,
)

print('Run output directories:')
for path in run_output_dirs:
    print(' -', path)

## 6) Quick sanity checks + summary

In [ ]:
from scripts.colab_bootstrap import quick_summary
quick_summary(DRIVE_STATE_PATH)

## 7) Optional: emergent scaffolding analysis + plots

In [ ]:
!python /content/Verdant-Minds/scripts/analysis/scaffolding_from_state.py --state "$DRIVE_STATE_PATH" --topk 6 --trials 500

print('Plots saved next to state file by default:')
print(' - emergent_scaffolding.png')
print(' - link_age_gaps.png')